# 📋 Extraction Autorisations — Ministère du Commerce Extérieur Algérien

**Objectif** : Lire le PDF d'autorisation (pages = images), exporter vers Excel, et enrichir avec les numéros de compte depuis le fichier banque.

**Matching en deux passes :**
1. **Exact** — nom identique après normalisation
2. **Fuzzy** — similarité la plus proche si pas de match exact

> ⚠️ La **page 1** (page de garde) est ignorée automatiquement.

## 1. Imports

In [ ]:
import json
import re
import time
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import Counter

import fitz  # PyMuPDF
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Pour le matching fuzzy — installé si absent
try:
    from rapidfuzz import process as fuzz_process, fuzz
    print("✅ rapidfuzz disponible")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "rapidfuzz", "--quiet"])
    from rapidfuzz import process as fuzz_process, fuzz
    print("✅ rapidfuzz installé")

print('✅ Imports OK')

## 2. Config  ⬅️ À MODIFIER CHAQUE MOIS

In [ ]:
# ──────────────────────────────────────────────────────────────
#  ⬇️  MODIFIER ICI CHAQUE MOIS
# ──────────────────────────────────────────────────────────────
PDF_PATH   = "/mnt/data/autorisation_ministere_fevrier2026.pdf"  # ← PDF autorisations
BANQUE_PATH = "/mnt/data/fichier_banque.xlsx"                    # ← fichier banque
PERIODE    = "Février 2026"                                       # ← période

# Noms des colonnes dans le fichier banque (adapter si différent)
COL_NOM_BANQUE    = "NOM_PRENOM"    # ← colonne nom complet dans le fichier banque
COL_COMPTE_BANQUE = "NUMERO_COMPTE" # ← colonne numéro de compte
# ──────────────────────────────────────────────────────────────

SKIP_FIRST_PAGE  = True   # Page 1 = page de garde → ignorée
FUZZY_THRESHOLD  = 70     # Score minimum (0-100) pour accepter un match fuzzy

OUTPUT_DIR     = Path("/mnt/data/transferts_out")
MODEL_PATH     = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 2000
IMAGE_MAX_SIZE = 2024
PDF_ZOOM       = 3.0
GPU_BATCH_SIZE = 8

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
date_str     = datetime.now().strftime("%Y%m%d_%H%M")
periode_slug = PERIODE.replace(" ", "_").replace("/", "-")
EXCEL_PATH   = OUTPUT_DIR / f"autorisations_{periode_slug}_{date_str}.xlsx"

print(f"Device          : {DEVICE}")
print(f"PDF             : {PDF_PATH}")
print(f"Fichier banque  : {BANQUE_PATH}")
print(f"Période         : {PERIODE}")
print(f"Fuzzy threshold : {FUZZY_THRESHOLD}%")
print(f"Excel output    : {EXCEL_PATH}")

## 3. Chargement fichier banque

In [ ]:
def normaliser_nom(nom: str) -> str:
    """Normalise un nom pour la comparaison : majuscules, sans accents, espaces normalisés."""
    if not nom:
        return ""
    # Supprimer accents
    nom = unicodedata.normalize('NFD', str(nom))
    nom = ''.join(c for c in nom if unicodedata.category(c) != 'Mn')
    # Majuscules, espaces multiples → simple
    nom = re.sub(r'\s+', ' ', nom.upper().strip())
    return nom


def charger_banque(path: str, col_nom: str, col_compte: str) -> dict:
    """
    Charge le fichier banque et retourne un dict :
      { nom_normalise : {"nom_original": ..., "numero_compte": ...} }
    """
    wb  = openpyxl.load_workbook(path)
    ws  = wb.active
    headers = [str(c.value).strip() if c.value else "" for c in ws[1]]

    # Trouver les index des colonnes (insensible à la casse)
    headers_upper = [h.upper() for h in headers]
    try:
        idx_nom     = headers_upper.index(col_nom.upper())
        idx_compte  = headers_upper.index(col_compte.upper())
    except ValueError:
        print(f"⚠️  Colonnes disponibles dans le fichier banque : {headers}")
        raise ValueError(f"Colonne '{col_nom}' ou '{col_compte}' introuvable. Vérifier COL_NOM_BANQUE / COL_COMPTE_BANQUE dans Config.")

    banque = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        nom     = row[idx_nom]
        compte  = row[idx_compte]
        if nom and compte:
            cle = normaliser_nom(str(nom))
            banque[cle] = {
                "nom_original" : str(nom).strip(),
                "numero_compte": str(compte).strip(),
            }
    return banque


banque_dict = charger_banque(BANQUE_PATH, COL_NOM_BANQUE, COL_COMPTE_BANQUE)
banque_keys = list(banque_dict.keys())  # liste des noms normalisés pour fuzzy

print(f"✅ Fichier banque chargé : {len(banque_dict)} employés")
print(f"   Exemples : {banque_keys[:5]}")

## 4. Fonction de matching (exact + fuzzy)

In [ ]:
def trouver_compte(nom_autorisation: str) -> dict:
    """
    Cherche le numéro de compte pour un nom venant du PDF autorisations.
    Retourne un dict avec :
      - numero_compte
      - match_type   : 'EXACT' | 'FUZZY' | 'NON TROUVÉ'
      - nom_banque   : nom tel qu'il apparaît dans le fichier banque
      - score        : score fuzzy (100 si exact, None si non trouvé)
    """
    if not nom_autorisation:
        return {"numero_compte": None, "match_type": "NON TROUVÉ", "nom_banque": None, "score": None}

    nom_norm = normaliser_nom(nom_autorisation)

    # ── Passe 1 : match exact ──
    if nom_norm in banque_dict:
        entry = banque_dict[nom_norm]
        return {
            "numero_compte": entry["numero_compte"],
            "match_type"   : "EXACT",
            "nom_banque"   : entry["nom_original"],
            "score"        : 100,
        }

    # ── Passe 2 : match fuzzy ──
    result = fuzz_process.extractOne(
        nom_norm,
        banque_keys,
        scorer=fuzz.token_sort_ratio,  # insensible à l'ordre des mots
        score_cutoff=FUZZY_THRESHOLD,
    )

    if result:
        best_key, score, _ = result
        entry = banque_dict[best_key]
        return {
            "numero_compte": entry["numero_compte"],
            "match_type"   : "FUZZY",
            "nom_banque"   : entry["nom_original"],
            "score"        : round(score),
        }

    # ── Non trouvé ──
    return {"numero_compte": None, "match_type": "NON TROUVÉ", "nom_banque": None, "score": None}


# Test rapide
test_nom = banque_keys[0] if banque_keys else "GULHAN ZAFER"
print(f"Test matching sur '{test_nom}' :")
print(trouver_compte(test_nom))
print("\n✅ Fonction matching OK")

## 5. Chargement modèle VLM

In [ ]:
print(f"Chargement Qwen2.5-VL sur {DEVICE}...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"✅ Modèle chargé en {time.time()-t0:.1f}s")

## 6. Conversion PDF → Images

In [ ]:
def pdf_to_images(pdf_path: str, skip_first: bool = True):
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    images = []
    mat = fitz.Matrix(PDF_ZOOM, PDF_ZOOM)
    start_idx = 1 if skip_first else 0
    for page_idx in range(start_idx, total_pages):
        pix = doc[page_idx].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        w, h = img.size
        if max(w, h) > IMAGE_MAX_SIZE:
            scale = IMAGE_MAX_SIZE / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        images.append((page_idx + 1, img))
    doc.close()
    return images, total_pages

print("Conversion PDF en images...")
images, total_pdf_pages = pdf_to_images(PDF_PATH, skip_first=SKIP_FIRST_PAGE)
print(f"✅ PDF total : {total_pdf_pages} pages | Pages traitées : {len(images)} (page {images[0][0]} → {images[-1][0]})")
display(images[0][1].resize((400, int(400 * images[0][1].height / images[0][1].width))))

## 7. Prompt + Extraction VLM

In [ ]:
PROMPT = f"""Tu analyses un document officiel algérien du Ministère du Commerce Extérieur.
La page contient un tableau avec les colonnes :
- Nom et prénom du travailleur expatrié (colonne "Le service Importé" ou similaire)
- Montant du salaire en DZD (colonne "Salaire" ou "القيمة الخدمة")
- Pays de provenance (colonne "Pays" ou "البلد المصدر")
- Devise (colonne "العملة" — généralement DZD)

Extrais TOUTES les lignes du tableau visibles sur cette page.
Retourne UNIQUEMENT un JSON valide, sans texte avant ou après :

{{
  "periode": "{PERIODE}",
  "lignes": [
    {{
      "nom_prenom": "NOM PRENOM",
      "montant_dzd": 488299.00,
      "pays": "TURQUIE",
      "devise": "DZD"
    }}
  ]
}}

Règles :
- nom_prenom : en majuscules, tel qu'écrit dans le document
- montant_dzd : nombre décimal (pas de guillemets), point comme séparateur décimal
- Si une cellule est illisible, mets null
- N'invente aucune donnée
- Inclus TOUTES les lignes visibles, même partielles
"""

def extract_page(image: Image.Image, page_num_pdf: int) -> list:
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": PROMPT},
    ]}]
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text_input], images=[image], return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    response  = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
    response  = re.sub(r"```json|```", "", response).strip()
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if match:
        response = match.group(0)
    try:
        data   = json.loads(response)
        lignes = data.get("lignes", [])
        for l in lignes:
            l["page_pdf"] = page_num_pdf
        return lignes
    except json.JSONDecodeError:
        print(f"  ⚠️  Page PDF {page_num_pdf} — JSON invalide : {response[:300]}")
        return []


# ── Lancement extraction ──
all_lignes = []
total   = len(images)
t_start = time.time()

for i in range(0, total, GPU_BATCH_SIZE):
    batch = images[i:i + GPU_BATCH_SIZE]
    print(f"\n🔄 Pages PDF {batch[0][0]}–{batch[-1][0]}  (batch {i//GPU_BATCH_SIZE + 1})")
    for page_num_pdf, img in batch:
        lignes = extract_page(img, page_num_pdf)
        all_lignes.extend(lignes)
        print(f"   Page PDF {page_num_pdf:>3} : {len(lignes):>3} lignes  |  Total : {len(all_lignes)}")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

elapsed = time.time() - t_start
print(f"\n✅ Extraction terminée — {len(all_lignes)} employés en {elapsed:.1f}s")

## 8. Enrichissement — Matching numéros de compte

In [ ]:
print("Matching numéros de compte...")

stats = Counter()

for ligne in all_lignes:
    result = trouver_compte(ligne.get("nom_prenom", ""))
    ligne["numero_compte"] = result["numero_compte"]
    ligne["match_type"]    = result["match_type"]
    ligne["nom_banque"]    = result["nom_banque"]
    ligne["score_match"]   = result["score"]
    stats[result["match_type"]] += 1

print(f"\n📊 Résultats matching :")
print(f"   ✅ EXACT       : {stats['EXACT']:>4} employés")
print(f"   🔶 FUZZY       : {stats['FUZZY']:>4} employés")
print(f"   ❌ NON TROUVÉ  : {stats['NON TROUVÉ']:>4} employés")
print(f"   ─────────────────────────")
print(f"   Total          : {len(all_lignes):>4} employés")

# Détail des matchs fuzzy pour vérification
fuzzy_list = [l for l in all_lignes if l["match_type"] == "FUZZY"]
if fuzzy_list:
    print(f"\n🔶 Détail matchs FUZZY (à vérifier) :")
    print(f"   {'NOM AUTORISATION':<35} {'NOM BANQUE':<35} {'SCORE':>6}")
    print("   " + "-" * 80)
    for l in fuzzy_list:
        nom_auto  = str(l.get('nom_prenom', '') or '')[:34]
        nom_banq  = str(l.get('nom_banque', '') or '')[:34]
        score     = l.get('score_match', '')
        print(f"   {nom_auto:<35} {nom_banq:<35} {str(score):>6}%")

# Détail des non trouvés
non_trouves = [l for l in all_lignes if l["match_type"] == "NON TROUVÉ"]
if non_trouves:
    print(f"\n❌ Noms NON TROUVÉS dans le fichier banque :")
    for l in non_trouves:
        print(f"   - {l.get('nom_prenom', '')}  (page PDF {l.get('page_pdf', '')})")

## 9. Export Excel

In [ ]:
def export_excel(lignes: list, output_path: Path, periode: str):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Autorisations"

    # ── Styles ──
    thin         = Side(style="thin")
    border       = Border(left=thin, right=thin, top=thin, bottom=thin)
    header_font  = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell_align   = Alignment(horizontal="left",   vertical="center")
    amt_align    = Alignment(horizontal="right",  vertical="center")
    ctr_align    = Alignment(horizontal="center", vertical="center")
    alt_fill     = PatternFill("solid", fgColor="DEEAF1")
    total_fill   = PatternFill("solid", fgColor="BDD7EE")
    # Couleurs match
    exact_fill   = PatternFill("solid", fgColor="E2EFDA")  # vert clair
    fuzzy_fill   = PatternFill("solid", fgColor="FFF2CC")  # jaune clair
    notfound_fill= PatternFill("solid", fgColor="FCE4D6")  # rouge clair

    # ── Titre ──
    ws.merge_cells("A1:J1")
    tc = ws["A1"]
    tc.value     = f"Autorisations Ministère du Commerce Extérieur — {periode}"
    tc.font      = Font(name="Calibri", bold=True, size=13, color="1F4E79")
    tc.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 30

    # ── Sous-titre ──
    ws.merge_cells("A2:J2")
    sc = ws["A2"]
    n_exact = sum(1 for l in lignes if l.get('match_type') == 'EXACT')
    n_fuzzy = sum(1 for l in lignes if l.get('match_type') == 'FUZZY')
    n_nf    = sum(1 for l in lignes if l.get('match_type') == 'NON TROUVÉ')
    sc.value     = (
        f"Extrait le {datetime.now().strftime('%d/%m/%Y %H:%M')} — {len(lignes)} employés  |  "
        f"✅ Exact: {n_exact}   🔶 Fuzzy: {n_fuzzy}   ❌ Non trouvé: {n_nf}"
    )
    sc.font      = Font(name="Calibri", italic=True, size=10, color="595959")
    sc.alignment = Alignment(horizontal="center")

    # ── Légende ──
    ws.merge_cells("A3:J3")
    leg = ws["A3"]
    leg.value     = "Couleurs : vert = match exact   |   jaune = match fuzzy (à vérifier)   |   rouge = non trouvé"
    leg.font      = Font(name="Calibri", italic=True, size=9, color="595959")
    leg.alignment = Alignment(horizontal="center")

    # ── En-têtes (10 colonnes) ──
    headers    = ["N°", "Nom & Prénom (Autorisation)", "Montant Autorisé (DZD)",
                  "Devise", "Pays", "Page PDF", "Période",
                  "N° Compte", "Nom Banque (match)", "Type Match"]
    col_widths = [5,    32,                            22,
                  8,    10,   9,        12,
                  20,   32,                    12]
    for col, (h, w) in enumerate(zip(headers, col_widths), 1):
        cell           = ws.cell(row=4, column=col, value=h)
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = header_align
        cell.border    = border
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[4].height = 30

    # ── Données ──
    total_montant = 0.0
    for idx, ligne in enumerate(lignes, 1):
        row = idx + 4

        # Couleur de la ligne selon type match
        match_type = ligne.get("match_type", "NON TROUVÉ")
        if match_type == "EXACT":
            row_fill = exact_fill
        elif match_type == "FUZZY":
            row_fill = fuzzy_fill
        else:
            row_fill = notfound_fill

        montant = ligne.get("montant_dzd")
        if montant is not None:
            try:
                montant = float(str(montant).replace(" ", "").replace(",", "."))
                total_montant += montant
            except:
                montant = None

        values = [
            idx,
            ligne.get("nom_prenom", ""),
            montant,
            ligne.get("devise", "DZD"),
            ligne.get("pays", "TURQUIE"),
            ligne.get("page_pdf", ""),
            periode,
            ligne.get("numero_compte", "") or "",
            ligne.get("nom_banque", "")    or "",
            match_type,
        ]
        for col, val in enumerate(values, 1):
            cell        = ws.cell(row=row, column=col, value=val)
            cell.border = border
            cell.fill   = row_fill
            if col == 3 and val is not None:
                cell.number_format = '#,##0.00'
                cell.alignment     = amt_align
            elif col in (1, 6, 10):
                cell.alignment = ctr_align
            else:
                cell.alignment = cell_align

    # ── Ligne TOTAL ──
    total_row = len(lignes) + 5
    ws.merge_cells(f"A{total_row}:B{total_row}")
    t_label = ws.cell(row=total_row, column=1, value="TOTAL")
    t_label.font = Font(bold=True, size=11)
    t_label.alignment = ctr_align
    t_label.border = border
    t_label.fill = total_fill
    t_amt = ws.cell(row=total_row, column=3, value=total_montant)
    t_amt.font = Font(bold=True, color="1F4E79", size=11)
    t_amt.number_format = '#,##0.00'
    t_amt.alignment = amt_align
    t_amt.fill = total_fill
    t_amt.border = border
    for col in range(4, 11):
        ws.cell(row=total_row, column=col).border = border
        ws.cell(row=total_row, column=col).fill   = total_fill

    ws.freeze_panes = "A5"
    wb.save(output_path)
    print(f"\n✅ Excel sauvegardé : {output_path}")
    print(f"   Total employés  : {len(lignes)}")
    print(f"   Total DZD       : {total_montant:,.2f}")


export_excel(all_lignes, EXCEL_PATH, PERIODE)

## 10. Vérification finale

In [ ]:
import os
size_kb = os.path.getsize(EXCEL_PATH) / 1024
print(f"✅ Fichier Excel généré")
print(f"   Chemin  : {EXCEL_PATH}")
print(f"   Taille  : {size_kb:.1f} KB")
print(f"\n🎯 Colonnes Excel :")
print(f"   A  — N°")
print(f"   B  — Nom & Prénom (document autorisation)")
print(f"   C  — Montant Autorisé (DZD)")
print(f"   D  — Devise")
print(f"   E  — Pays")
print(f"   F  — Page PDF source")
print(f"   G  — Période")
print(f"   H  — N° Compte bancaire  ← issu du fichier banque")
print(f"   I  — Nom banque (tel que dans le fichier banque)")
print(f"   J  — Type match (EXACT / FUZZY / NON TROUVÉ)")
print(f"\n🎨 Code couleur :")
print(f"   Vert   — match exact")
print(f"   Jaune  — match fuzzy (à valider manuellement)")
print(f"   Rouge  — aucun match trouvé")